In [ ]:
import jax
import jax.numpy as jnp

import numpy as np

from flax import nnx

from probjax.nn import GaussianFourierEmbedding, Transformer
from probjax.nn.loss_fn.denoising import build_time_dependent_denoising_loss
from probjax.nn.nets.diffusion_model import EDM
from probjax.nn import MLP



class VectorTokenizer(nnx.Module, experimental_pytree=True):
    def __init__(self, num_nodes, dims, rngs, value_dim=20, id_dim=20, cond_dim=10):
        self.model_dim = value_dim + id_dim + cond_dim
        self.num_nodes = num_nodes
        self.max_dim = max(dims)
        self.dims = list(dims)
        self.value_dim = value_dim
        self.id_dim = id_dim
        self.cond_dim = cond_dim

        self.embed_id = nnx.Embed(num_nodes, self.id_dim, rngs=rngs)
        self.embed_value = nnx.Linear(self.max_dim, self.value_dim, rngs=rngs)
        self.outlayer = nnx.Linear(self.model_dim, self.max_dim, rngs=rngs)
        if cond_dim > 0:
            self.condition_token = nnx.Param(
                0.01 * jax.random.normal(rngs.next(), (1, cond_dim))
            )

    def __call__(self, x, node_ids, condition_mask):
        node_embed = self.embed_id(node_ids)
        # Pad all inputs to max_dim
        x = jax.tree_map(
            lambda x: jnp.pad(
                x, [(0, 0)] * (x.ndim - 1) + [(0, self.max_dim - x.shape[-1])]
            ),
            x,
        )
        x = jnp.concatenate(x, axis=-2)

        value_embed = self.embed_value(x)

        if self.cond_dim > 0:
            condition_embed = self.condition_token.value * condition_mask[..., None]
            input_embed = jnp.concatenate(
                [node_embed, value_embed, condition_embed], axis=-1
            )
        else:
            input_embed = jnp.concatenate([node_embed, value_embed], axis=-1)
        return input_embed

    def decode(self, h):
        out = self.outlayer(h)
        out_split = jnp.split(out, self.num_nodes, axis=-2)
        out = jax.tree_map(lambda x, d: x[..., :d], out_split, self.dims)
        return out

    


class PyTreeTokenizer(nnx.Module, experimental_pytree=True):
    def __init__(self, nets, id_dim=20, cond_dim=10):
        self.nets = nets
        self.embed_id = nnx.Embed(len(self.nets), id_dim)

    def __call__(self, x, node_ids, context=None, condition_mask=None):
        net_subs = [self.nets[i] for i in node_ids]
        x = jax.tree_util.tree_map(lambda x, net: net(x), x, net_subs)
        x = jax.tree_util.tree_map(lambda x: x[...,None,:], x)
        val_embeding = jnp.concatenate(list(x.values()), axis=-2)

        ids = self.embed_id(jnp.array(node_ids, dtype=jnp.int32))

        embeddings = jnp.concatenate([val_embeding, ids], axis=-1)
        if context is not None:
            embeddings += context
        
        return embeddings



In [ ]:
x1 = np.random.randn(10, 5)
x2 = np.random.randn(10, 2)
x3 = np.random.randn(10, 3)

data = {"x1": x1, "x2": x2, "x3": x3}   

In [13]:
tree.node_data()

(dict, ['x1', 'x2', 'x3'])

In [9]:
tree = jax.tree_structure(data)
mlp1 = MLP([5, 50, 50], rngs=nnx.Rngs(0))
mlp2 = MLP([2, 50, 50], rngs=nnx.Rngs(0))
mlp3 = MLP([3, 50, 50], rngs=nnx.Rngs(0))

net_tree = jax.tree_util.tree_unflatten(tree, [mlp1, mlp2, mlp3])

/tmp/ipykernel_8514/1417613613.py:1: DeprecationWarning: jax.tree_structure is deprecated: use jax.tree.structure (jax v0.4.25 or newer) or jax.tree_util.tree_structure (any JAX version).
  tree = jax.tree_structure(data)
2024-11-05 17:50:57.802307: W external/xla/xla/service/gpu/nvptx_compiler.cc:930] The NVIDIA driver's CUDA version is 12.5 which is older than the PTX compiler version 12.6.77. Because the driver is older than the PTX compiler version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.


In [10]:
net_tree

{'x1': MLP(
   layers=[Linear(
     kernel=Param(
       value=Array(shape=(5, 50), dtype=float32)
     ),
     bias=Param(
       value=Array(shape=(50,), dtype=float32)
     ),
     in_features=5,
     out_features=50,
     use_bias=True,
     dtype=None,
     param_dtype=<class 'jax.numpy.float32'>,
     precision=None,
     kernel_init=<function variance_scaling.<locals>.init at 0x7f391887c220>,
     bias_init=<function zeros at 0x7f39194882c0>,
     dot_general=<function dot_general at 0x7f3919cc91c0>
   ), Linear(
     kernel=Param(
       value=Array(shape=(50, 50), dtype=float32)
     ),
     bias=Param(
       value=Array(shape=(50,), dtype=float32)
     ),
     in_features=50,
     out_features=50,
     use_bias=True,
     dtype=None,
     param_dtype=<class 'jax.numpy.float32'>,
     precision=None,
     kernel_init=<function variance_scaling.<locals>.init at 0x7f391887c220>,
     bias_init=<function zeros at 0x7f39194882c0>,
     dot_general=<function dot_general at 0x7f391